# 🎤 DSM-ASR v3: True Delayed Streams (Moshi Paper)

**Architecture from [arXiv:2410.00037](https://arxiv.org/abs/2410.00037)**:
- **Mimi** encodes audio → 8 codebooks at 12.5Hz
- **Qwen3-0.6B** processes parallel audio+text streams
- Text delayed ~2s behind audio (like Moshi)
- Word-level timestamps from **whisper-timestamped**

---
**Runtime**: A100 40GB

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone repo
REPO_URL = "https://github.com/AhPro7/DSM.git"
!rm -rf /content/DSM
!git clone {REPO_URL} /content/DSM
%cd /content/DSM

In [ ]:
# Install dependencies
!pip install -q torch transformers>=4.51.0 datasets[audio] accelerate \
    soundfile librosa tqdm jiwer wandb whisper-timestamped

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
login()  # Or: login(token="hf_YOUR_TOKEN")

## 2. Preprocess — Mimi + Word Timestamps
Encodes audio with Mimi AND generates word-level timestamps with Whisper

In [ ]:
# Test with 20 samples first
!python data/prepare_data.py --max_samples 20

In [ ]:
# Verify preprocessed data
import json, numpy as np
with open("preprocessed_data/manifest.json") as f:
    m = json.load(f)
print(f"Processed: {m['total_processed']}, Errors: {m['total_errors']}")
if m['samples']:
    s = m['samples'][0]
    print(f"Sample: {s['num_frames']} frames, {s['duration']:.1f}s, {s['num_words']} words")
    print(f"Text: {s['text'][:100]}")
    # Check timestamps
    d = np.load(s['path'], allow_pickle=True)
    print(f"Timestamps: {list(zip(d['word_texts'][:3], d['word_starts'][:3]))}")

In [ ]:
# Full dataset preprocessing (uncomment when ready)
# !python data/prepare_data.py

## 3. Sanity Checks

In [ ]:
# Test dataset
!python data/dataset.py

In [ ]:
# Test model
!python model/dsm_asr.py

## 4. Train
Shows sample predictions with WER during training.
Audio+text are parallel streams — text delayed ~2s behind audio.

In [ ]:
# Quick sanity run (10 steps)
!rm -rf output
!python train.py --max_steps 10 --batch_size 2 --max_samples 20

In [ ]:
# Full training
!rm -rf output preprocessed_data
!python data/prepare_data.py
!python train.py --batch_size 4 --num_epochs 15

In [ ]:
# Plot loss
import json, matplotlib.pyplot as plt
try:
    with open("output/training_log.json") as f:
        log = json.load(f)
    plt.figure(figsize=(10, 4))
    plt.plot([e['step'] for e in log], [e['loss'] for e in log])
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss')
    plt.grid(True, alpha=0.3); plt.show()
except: print('No log yet')

## 5. Evaluate

In [ ]:
!python evaluate.py --checkpoint output/best --output output/eval_results.json

## 6. Inference

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
!python inference.py --checkpoint output/best --audio {audio_file}

## 7. Save Model

In [ ]:
# Save to HuggingFace Hub
from huggingface_hub import HfApi
api = HfApi()
repo_id = "nadsoft/dsm-asr-arabic"  # Change this
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="output/best", repo_id=repo_id)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")

In [ ]:
# Backup to Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r output/best /content/drive/MyDrive/dsm_asr_checkpoint/
print('✅ Saved to Drive')